# Physics-Informed Neural Networks for Unsteady 2D Heat Transfer
## Simulating Meat Cooking on a Weber Grill
---
**Module:** Computational Fluid Dynamics  
**Department:** Mechanical & Aeronautical Engineering  
**References:** Raissi, M., Yazdani, A., & Karniadakis, G.E. (2020). *Hidden fluid mechanics.* Science, 367(6481), 1026–1030. https://doi.org/10.1126/science.aaw4741  
Raissi, M., Perdikaris, P., Karniadakis, G. E. (2019). Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. *Journal of Computational Physics*, 378, 686-707. https://doi.org/10.1016/j.jcp

---
### Learning Objectives
By the end of this notebook you will be able to:
1. Explain how a PINN differs from a standard data-driven ANN
2. Understand how the heat equation is embedded in the PINN loss via automatic differentiation
3. Train both models in PyTorch and interpret training and validation loss curves
4. Benchmark PINN and ANN predictions against FDM and an exact Fourier solution
5. Systematically investigate hyperparameter effects and explain the results
6. Evaluate model generalisation to an unseen material (lamb)

### Problem Statement
We solve the **2D unsteady heat conduction equation** on a rectangular domain
representing a cross-section of meat on a Weber grill (lid closed):

$$\rho c_p \, \frac{\partial T}{\partial t} = k \left(\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2}\right)$$

### Thermal Simulation Parameters

| Quantity | Value |
| :--- | :--- |
| **Domain** | $x \in [0, 0.05\,\text{m}]$, $y \in [0, 0.15\,\text{m}]$, $t \in [0, 894\,\text{s}]$ |
| **Initial Condition** | $T(x,y,0) = 288.15\,\text{K}$ (Fridge temp, ~15°C) |
| **Boundary Conditions** | $T = 473.15\,\text{K}$ on all four walls (Grill surface, ~200°C) |
| **Material Inputs** | $\rho, c_p, k$ — Density, specific heat, and conductivity |

### Data Strategy
| Split | Contents | Purpose |
|-------|---------|---------|
| Train | 80% of combined beef/chicken/pork data | ANN weight updates |
| Validation | 20% of combined beef/chicken/pork data | Monitor overfitting |
| Test | Lamb (never seen during training) | Final generalisation evaluation |


In [ ]:
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    XMAX, YMAX, T_MAX, NX, NY, NT, DT,
    T_BOUNDARY, T_INITIAL, T_SAFE, DELTA_T,
    MEAT_PROPERTIES, TEST_MEAT,
)
from utils.dataset    import (build_ann_dataloaders, sample_collocation_points,
                               load_meat_data, normalise_inputs)
from utils.models     import ANN, PINN
from utils.training   import train_ann, train_pinn
from utils.evaluation import (evaluate_model, plot_loss_curves,
                               plot_field_comparison, plot_centre_temperature,
                               print_metrics_table)
from utils.fourier    import T_fourier_grid, T_fourier_centre_history
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GLOBAL_SEED = 42

def set_seed(seed=GLOBAL_SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(GLOBAL_SEED)
print(f"Global seed : {GLOBAL_SEED} (call set_seed() to reset before each experiment)")
print(f"Device  : {device}")
print(f"Device Name   : {torch.cuda.get_device_name(device)}")
print(f"PyTorch : {torch.__version__}")


---
## Section 2: Reference Data

In [ ]:
# ── Load FDM reference datasets ───────────────────────────────────────────────
DATA_DIR = "data"
fdm_data = {}

print("Loading FDM reference datasets...")
for meat in MEAT_PROPERTIES:
    raw   = load_meat_data(DATA_DIR, meat)
    props = MEAT_PROPERTIES[meat]
    nx_int, ny_int = NX - 2, NY - 2
    n_spatial = nx_int * ny_int

    x_grid = np.linspace(0, XMAX, NX)
    y_grid = np.linspace(0, YMAX, NY)
    t_grid = np.arange(NT) * DT

    u = np.full((NX, NY, NT), T_BOUNDARY)
    for it in range(NT):
        u[1:-1, 1:-1, it] = raw['T'][it*n_spatial:(it+1)*n_spatial].reshape(nx_int, ny_int)

    fdm_data[meat] = {'u': u, 'x': x_grid, 'y': y_grid, 't': t_grid,
                       'rho': props['rho'], 'cp': props['cp'], 'k': props['k']}
    alpha = props['k'] / (props['rho'] * props['cp'])
    print(f"  {meat:8s}: alpha={alpha:.3e} m²/s | T_centre(final)={u[NX//2,NY//2,-1]:.2f} K")


In [ ]:
# ── Visualise reference fields ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (meat, sol) in zip(axes, fdm_data.items()):
    im = ax.contourf(sol['y']*100, sol['x']*100, sol['u'][:,:,-1],
                     30, cmap='hot', vmin=T_INITIAL, vmax=T_BOUNDARY)
    plt.colorbar(im, ax=ax, label='T [K]')
    alpha = sol['k'] / (sol['rho'] * sol['cp'])
    ax.set_title(f"{meat.capitalize()} | α={alpha:.2e} m²/s", fontsize=10)
    ax.set_xlabel('y [cm]'); ax.set_ylabel('x [cm]')
plt.suptitle('FDM Reference — Final Temperature Field (t = 894 s)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Centre temperature vs time
plt.figure(figsize=(9, 5))
for meat, sol in fdm_data.items():
    alpha = sol['k'] / (sol['rho'] * sol['cp'])
    T_exact_c = T_fourier_centre_history(sol['t'], alpha)
    plt.plot(sol['t'], sol['u'][NX//2, NY//2, :], lw=2, label=f"{meat.capitalize()} FDM")
    plt.plot(sol['t'], T_exact_c, '--', lw=1.2, alpha=0.7, label=f"{meat.capitalize()} Exact")
plt.axhline(T_SAFE, color='red', ls=':', lw=1.5, label=f'Safe temp. {T_SAFE-273.15:.0f}°C', alpha=0.7)
plt.xlabel('Time [s]', fontsize=12); plt.ylabel('Centre Temperature [K]', fontsize=12)
plt.title('Centre Temperature — FDM vs Exact (Fourier)', fontsize=11)
plt.legend(fontsize=9, ncol=2); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


---
## Section 3: Data Preparation

### 3.1 Input Normalisation
All six inputs are mapped to $[0, 1]$ before entering the network.

| Input | Normalisation |
|-------|--------------|
| $x$ | $\hat{x} = x / L_x$ |
| $y$ | $\hat{y} = y / L_y$ |
| $t$ | $\hat{t} = t / t_{\\max}$ |
| $\rho$, $c_p$, $k$ | min-max over training meat range |

### 3.2 Train / Validation Split
All three training meat datasets are combined into one pool, shuffled,
and split into training and validation subsets. The split ratio is
controlled by `VAL_SPLIT` in the hyperparameter block.

A validation loss that **diverges** from the training loss indicates overfitting.

### 3.3 Test Set
**Lamb** is never loaded during training or validation. It is held out
entirely and only used in Section 8 to evaluate generalisation.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  HYPERPARAMETERS — modify these in Section 7 experiments
# ═══════════════════════════════════════════════════════════════════════════════

N_HIDDEN      = 5        # Number of hidden layers
N_NEURONS     = 40       # Neurons per hidden layer

LR            = 1e-3     # Adam learning rate
EPOCHS_ANN    = 5000     # ANN training epochs
EPOCHS_ADAM   = 10000    # Adam phase epochs
EPOCHS_LBFGS  = 500      # L-BFGS iterations (set 0 to skip)

# PINN loss weights — λ_bc and λ_ic higher than λ_pde so BCs/ICs are
# enforced before the PDE residual is minimised (prevents trivial solution)
LAMBDA_PDE    = 1.0      # Weight on PDE residual loss
LAMBDA_BC     = 10.0     # Weight on boundary condition loss
LAMBDA_IC     = 10.0     # Weight on initial condition loss

N_PER_MEAT    = 10000    # Data points sampled per meat before split
VAL_SPLIT     = 0.2      # Fraction used for validation (80/20 split)
BATCH_SIZE    = 512      # ANN mini-batch size

# ═══════════════════════════════════════════════════════════════════════════════

print("Building ANN DataLoaders...")
train_loader, val_loader, n_train, n_val = build_ann_dataloaders(
    data_dir   = DATA_DIR,
    meat_names = list(MEAT_PROPERTIES.keys()),
    n_per_meat = N_PER_MEAT,
    val_split  = VAL_SPLIT,
    batch_size = BATCH_SIZE,
    device     = device,
    seed       = 42,
)

print()
col, col_val = sample_collocation_points(
    N_col=10000, N_bc=800, N_ic=800,
    N_col_val=2000, N_bc_val=200, N_ic_val=200,
    seed=42,
)


---
## Section 4: Network Architectures

Both the ANN and PINN share **identical architecture**. The only difference
is the loss function — which is the central insight of this project.

| | ANN | PINN |
|--|-----|------|
| Architecture | Identical | Identical |
| Loss | MSE vs FDM data | PDE + BC + IC residuals |
| Training data | Required | Not required |
| Physics enforced | No | Yes (via autograd) |

### Why `tanh`?
The PINN computes second-order spatial derivatives of the network output
via automatic differentiation. The activation function must be at least
twice differentiable. `tanh` satisfies this. `ReLU` does not — its second
derivative is zero almost everywhere, making the PDE residual uninformative.

### PINN Loss

$$\mathcal{L} = \lambda_{\text{pde}}\mathcal{L}_{\text{pde}} + \lambda_{\text{bc}}\mathcal{L}_{\text{bc}} + \lambda_{\text{ic}}\mathcal{L}_{\text{ic}}$$

**PDE residual** at interior collocation points via autograd:

$$\mathcal{L}_{\text{pde}} = \frac{1}{N_c}\sum_{i=1}^{N_c}\left[ \frac{\rho c_p \frac{\partial T}{\partial t} - k \left( \frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2} \right)}{\frac{\rho c_p \Delta T}{t_{\max}}} \right]^2$$

**Chain rule for normalized coordinates:**

$$\frac{\partial T}{\partial t} = \frac{\partial T}{\partial \hat{t}} \cdot \frac{1}{t_{\max}}, \qquad \frac{\partial^2 T}{\partial x^2} = \frac{\partial^2 T}{\partial \hat{x}^2} \cdot \frac{1}{L_x^2}$$


In [ ]:
ann  = ANN(n_hidden=N_HIDDEN, n_neurons=N_NEURONS).to(device)
pinn = PINN(n_hidden=N_HIDDEN, n_neurons=N_NEURONS,
             lambda_pde=LAMBDA_PDE, lambda_bc=LAMBDA_BC,
             lambda_ic=LAMBDA_IC).to(device)

print("=" * 45)
ann.architecture_summary()
print()
pinn.architecture_summary()
print("=" * 45)


---
## Section 5: Training

In [ ]:
# ── Train ANN ─────────────────────────────────────────────────────────────────
loss_ann = train_ann(ann, train_loader, val_loader,
                      epochs=EPOCHS_ANN, lr=LR, device=device)


In [ ]:
# ── Train PINN (two-phase: Adam → L-BFGS) ────────────────────────────────────
loss_pinn = train_pinn(pinn, col, col_val,
                        epochs_adam   = EPOCHS_ADAM,
                        lr            = LR,
                        epochs_lbfgs  = EPOCHS_LBFGS,
                        resample_every= 2000,
                        device        = device)


In [ ]:
# ── Loss curves — training and validation ─────────────────────────────────────
plot_loss_curves(loss_ann, loss_pinn,
                 lambda_pde=LAMBDA_PDE, lambda_bc=LAMBDA_BC, lambda_ic=LAMBDA_IC)


### Interpreting the Loss Curves

**ANN:** A gap between train and validation loss indicates overfitting —
the model has memorised the training data and does not generalise.
If the validation loss increases while training loss decreases, stop training earlier.

**PINN:** Train and validation losses use different (non-overlapping) collocation
points but the same physics. A large gap here is unusual — if it occurs, the
model may be fitting the specific random collocation points rather than the
underlying physics. Re-sampling collocation points or increasing `N_col` may help.


---
## Section 6: Evaluation and Benchmarking

In [ ]:
# ── Evaluate all training meats ───────────────────────────────────────────────
print("=" * 65)
eval_results = {}
for meat in MEAT_PROPERTIES:
    print()
    r_ann  = evaluate_model(ann,  meat, MEAT_PROPERTIES[meat],
                             fdm_data[meat], device=device)
    r_pinn = evaluate_model(pinn, meat, MEAT_PROPERTIES[meat],
                             fdm_data[meat], device=device)
    eval_results[meat] = {'ann': r_ann, 'pinn': r_pinn}
print("=" * 65)


In [ ]:
print_metrics_table(eval_results)

In [ ]:
for meat in MEAT_PROPERTIES:
    plot_field_comparison(meat, eval_results[meat]['ann'],
                           eval_results[meat]['pinn'],
                           fdm_data[meat]['x'], fdm_data[meat]['y'])


In [ ]:
for meat in MEAT_PROPERTIES:
    plot_centre_temperature(meat, MEAT_PROPERTIES[meat], fdm_data[meat],
                             models_dict={'ANN': ann, 'PINN': pinn}, device=device)


### Critical Note

The ANN outperforms the PINN on the training meats — this is expected. The ANN has seen this exact data; the PINN has not. Before drawing conclusions, proceed to Section 8 and evaluate both models on lamb, a material neither model has encountered. The results will reframe what you see here.

---
## Section 7: Hyperparameter Tuning  *(Student Task)*

### Background
Hyperparameters are set before training and control the network structure
and optimisation process. They are not learned — you must choose them
deliberately and justify your choices.

### How to run an experiment
Each experiment below has a clearly labelled code block. To run it:
1. Scroll to the **CHANGE THIS** line in the block
2. Update the single value indicated
3. Run the cell — training, evaluation and logging happen automatically
4. Read the printed output and note what changed

You do not need to rerun the baseline models. Each experiment creates a
fresh model independently.

### Experiments to run

| # | What you vary | Values to test |
|---|--------------|---------------|
| 1 | Network depth (`N_HIDDEN`) | 2, 5, 8 |
| 2 | Learning rate (`LR`) | 1e-2, 1e-3, 1e-4 |
| 3 | Loss weights (`LAMBDA_BC`, `LAMBDA_IC`) | 1, 10, 50 |

Run each experiment for all values listed. Use `print_experiment_log()`
at the end to produce your results table for the report.

### Reflection Questions

**Q1 — Network depth:**
How does changing `N_HIDDEN` affect the training loss curve and final accuracy?
At what depth does performance plateau or degrade? What does this reveal about
the representational capacity needed for this problem?

**Q2 — Learning rate:**
What happens when the learning rate is too high? Too low? Examine both the
loss curve shape and the final MAE. Why does Adam handle this differently
from a fixed learning rate method?

**Q3 — Loss weights:**
When `LAMBDA_BC = LAMBDA_IC = 1`, what happens to PINN accuracy?
When set to 50, what changes? Explain in terms of the competition between
the PDE residual and the boundary/initial condition terms during training.


In [ ]:
import torch.nn as nn

def run_experiment(model_type='PINN',
                   n_hidden=5, n_neurons=40,
                   lr=1e-3, epochs_adam=5000, epochs_lbfgs=200,
                   lambda_pde=1.0, lambda_bc=10.0, lambda_ic=10.0,
                   label=None):
    """
    Train a fresh model with the given hyperparameters and evaluate on beef.

    Returns
    -------
    model   : trained model
    history : loss history dict
    metrics : error metrics dict
    """
    if label is None:
        label = f"{model_type} | layers={n_hidden} | lr={lr:.0e} | λbc={lambda_bc}"
    set_seed(GLOBAL_SEED)  # Reset all seeds before each experiment
    print(f"\nExperiment: {label}")
    print("-" * 60)

    if model_type == 'PINN':
        model = PINN(n_hidden=n_hidden, n_neurons=n_neurons,
                      lambda_pde=lambda_pde, lambda_bc=lambda_bc,
                      lambda_ic=lambda_ic).to(device)
        history = train_pinn(model, col, col_val,
                              epochs_adam=epochs_adam, lr=lr,
                              epochs_lbfgs=epochs_lbfgs,
                              resample_every=2000,
                              print_every=max(1, epochs_adam//5),
                              device=device)
    else:
        model = ANN(n_hidden=n_hidden, n_neurons=n_neurons).to(device)
        history = train_ann(model, train_loader, val_loader,
                             epochs=epochs_adam, lr=lr,
                             print_every=max(1, epochs_adam//5),
                             device=device)

    result = evaluate_model(model, 'beef', MEAT_PROPERTIES['beef'],
                             fdm_data['beef'], device=device)
    return model, history, result['metrics']


experiment_log = []

def log_experiment(label, model, metrics, notes=''):
    """Record an experiment result."""
    experiment_log.append({'label': label, 'model': model, **metrics, 'notes': notes})
    print(f"Logged: {label}")

def print_experiment_log():
    if not experiment_log:
        print("No experiments recorded yet.")
        return
    header = f"{'#':<4} {'Label':<45} {'MAE vs FDM':>11} {'RMSE vs FDM':>12}  Notes"
    print(header); print("-" * len(header))
    for i, r in enumerate(experiment_log, 1):
        print(f"{i:<4} {r['label']:<45} {r.get('mae_fdm', float('nan')):>11.4f}"
              f" {r.get('rmse_fdm', float('nan')):>11.4f}   {r.get('notes', '')}")
    print(f"\nTo use experiment N as your best model, write:")
    print(f"  best_pinn = experiment_log[N-1]['model']")

print("Experiment runner defined. Proceed to the experiments below.")


### Experiment 1 — Network Depth
Run this cell three times, changing `N_HIDDEN_EXP` to **2**, then **5**, then **8**.

In [ ]:
# ── CHANGE THIS: add or remove values from the list ──────────────────────────
depth_values = [2, 5, 8]
# ─────────────────────────────────────────────────────────────────────────────

for n in depth_values:
    label = f"PINN | depth={n} layers"
    model_exp1, hist_exp1, met_exp1 = run_experiment(
        model_type='PINN', n_hidden=n, n_neurons=N_NEURONS,
        lr=LR, epochs_adam=EPOCHS_ADAM, epochs_lbfgs=EPOCHS_LBFGS,
        lambda_pde=LAMBDA_PDE, lambda_bc=LAMBDA_BC, lambda_ic=LAMBDA_IC,
        label=label)
    log_experiment(label, model_exp1, met_exp1, notes=f'N_HIDDEN={n}')

### Experiment 2 — Learning Rate
Run this cell three times, changing `LR_EXP` to **1e-2**, then **1e-3**, then **1e-4**.

In [ ]:
# ── CHANGE THIS ───────────────────────────────────────────────────────────────
lr_values = [1e-2, 1e-3, 1e-4]
# ─────────────────────────────────────────────────────────────────────────────

for lr_exp in lr_values:
    label = f"PINN | lr={lr_exp:.0e}"
    model_exp2, hist_exp2, met_exp2 = run_experiment(
        model_type='PINN', n_hidden=N_HIDDEN, n_neurons=N_NEURONS,
        lr=lr_exp, epochs_adam=EPOCHS_ADAM, epochs_lbfgs=EPOCHS_LBFGS,
        lambda_pde=LAMBDA_PDE, lambda_bc=LAMBDA_BC, lambda_ic=LAMBDA_IC,
        label=label)
    log_experiment(label, model_exp2, met_exp2, notes=f'LR={lr_exp:.0e}')

### Experiment 3 — Loss Weights
Run this cell three times, changing `LAMBDA_EXP` to **1**, then **10**, then **50**.

In [ ]:
# ── CHANGE THIS ───────────────────────────────────────────────────────────────
lambda_values = [1, 10, 50]
# ─────────────────────────────────────────────────────────────────────────────

for lam in lambda_values:
    label = f"PINN | λ_bc=λ_ic={lam}"
    model_exp3, hist_exp3, met_exp3 = run_experiment(
        model_type='PINN', n_hidden=N_HIDDEN, n_neurons=N_NEURONS,
        lr=LR, epochs_adam=EPOCHS_ADAM, epochs_lbfgs=EPOCHS_LBFGS,
        lambda_pde=LAMBDA_PDE, lambda_bc=lam, lambda_ic=lam,
        label=label)
    log_experiment(label, model_exp3, met_exp3, notes=f'lambda_bc=lambda_ic={lam}')


### Results Summary

In [ ]:
# ── Experiment results ────────────────────────────────────────────────────────
# Note: the variable names model_exp1, model_exp2, model_exp3 always hold the
# last model trained in each loop. To retrieve any specific model for Section 8,
# use the experiment log by number — e.g. experiment_log[2]['model'] retrieves
# experiment #3. Run this cell first to see which number corresponds to which
# configuration, then assign your best PINN in Section 8 accordingly.
# ─────────────────────────────────────────────────────────────────────────────
print_experiment_log()

---
## Section 8: Test Meat — Generalisation Evaluation  *(Student Task)*

Your models were trained on **beef, chicken, and pork**.  
Now evaluate your best models on **lamb** — never seen during training.

### Your Task
1. Replace `best_ann` and `best_pinn` with your best models from Section 7
2. Evaluate on lamb and compare errors to the training meat results
3. In your report, explain which model generalises better and why


In [ ]:
# ── Load lamb FDM reference ───────────────────────────────────────────────────
print("Loading test meat (lamb) reference...")
raw_lamb   = load_meat_data(DATA_DIR, 'lamb')
props_lamb = TEST_MEAT['lamb']
nx_int, ny_int = NX - 2, NY - 2
n_spatial = nx_int * ny_int

x_grid = np.linspace(0, XMAX, NX)
y_grid = np.linspace(0, YMAX, NY)
t_grid = np.arange(NT) * DT

u_lamb = np.full((NX, NY, NT), T_BOUNDARY)
for it in range(NT):
    u_lamb[1:-1, 1:-1, it] = raw_lamb['T'][it*n_spatial:(it+1)*n_spatial].reshape(nx_int, ny_int)

fdm_lamb = {'u': u_lamb, 'x': x_grid, 'y': y_grid, 't': t_grid, **props_lamb}
alpha_lamb = props_lamb['k'] / (props_lamb['rho'] * props_lamb['cp'])
print(f"Lamb: alpha={alpha_lamb:.3e} m²/s | T_centre(final)={u_lamb[NX//2,NY//2,-1]:.2f} K")


In [ ]:
# ── Substitute your best models from Section 7 ───────────────────────────────
# best_ann  — always use the baseline ANN from Section 5 (do not change).
#             The ANN is not tuned in Section 7; it serves as a fixed
#             data-driven reference to compare against your best PINN.
#
# best_pinn — replace 'pinn' with your best PINN from Section 7.
#             Use the experiment log to identify the best configuration:
#               1. Run print_experiment_log() above to see all results
#               2. Identify the experiment number with the lowest MAE vs FDM
#               3. Assign it here — e.g. for experiment #3:
#                  best_pinn = experiment_log[2]['model']
#
#             Note: experiment_log indices are zero-based, so experiment #N
#             is accessed as experiment_log[N-1]['model']
# ─────────────────────────────────────────────────────────────────────────────

best_ann  = ann                            # Fixed — Section 5 baseline (do not change). We are only tuning the PINN in Section 7, so we keep the ANN as a constant reference.

# CHANGE THIS: assign your best PINN from the experiment log (see instructions above)__________________
best_pinn = experiment_log[3]['model']     # Replace 0 with your best experiment index
# _____________________________________________________________________________________________________
print(f"Evaluating on lamb with:")
print(f"  best_ann  : {best_ann.__class__.__name__}  "
      f"({best_ann.n_hidden} layers × {best_ann.n_neurons} neurons)")
print(f"  best_pinn : {best_pinn.__class__.__name__} "
      f"({best_pinn.n_hidden} layers × {best_pinn.n_neurons} neurons)")

print("ANN on lamb:")
r_ann_lamb  = evaluate_model(best_ann,  'lamb', props_lamb, fdm_lamb, device=device)
print()
print("PINN on lamb:")
r_pinn_lamb = evaluate_model(best_pinn, 'lamb', props_lamb, fdm_lamb, device=device)


In [ ]:
plot_field_comparison('lamb', r_ann_lamb, r_pinn_lamb, x_grid, y_grid)

In [ ]:
plot_centre_temperature('lamb', props_lamb, fdm_lamb,
                         models_dict={'ANN': best_ann, 'PINN': best_pinn},
                         device=device)


---
### Reflection — Revisiting Section 6

In Section 6 you observed that the ANN outperformed the PINN on all three
training meats (beef, chicken, pork). This is expected — the ANN was trained
directly on data from those materials.

Now that you have evaluated both models on lamb, revisit that observation:

- Which model performed better on the unseen material?
- What does this tell you about the difference between learning from data
  versus learning from physics?
- The ANN has never seen lamb data, and neither has the PINN. Why do their
  generalisation errors differ so significantly?
- In an engineering context, which model would you trust more when applying
  it to a new material you have no data for? Why?

This contrast — strong performance on training conditions versus strong
generalisation to new conditions — is one of the central arguments for
physics-informed approaches in scientific machine learning. A model that
knows the governing equations does not need to have seen every possible
material to make physically consistent predictions.

Incorporate this reflection into your report discussion in Section 9.

---
## Section 9: Report Guidance

### 1. Model Understanding
- Describe the network architecture and the role of each hyperparameter
- Derive the chain rule used to compute physical derivatives from normalised-coordinate outputs
- Explain the composite PINN loss — what does each term enforce?
- Why is `tanh` required for PINN autograd?

### 2. Hyperparameter Study
- Present all experiments using `print_experiment_log()`
- For each hyperparameter: describe the trend and provide a physical/mathematical explanation
- Include train vs validation loss curves for each experiment — discuss any overfitting

### 3. Benchmarking — Training Meats
- Compare ANN, PINN, FDM, and Exact (MAE, RMSE) for all three training meats
- Plot temperature fields and centre temperature histories
- Discuss where each method performs best and worst

### 4. Generalisation — Test Meat (Lamb)
- Compare ANN and PINN on the unseen test material
- Which generalises better, and why?

### 5. Critical Method Comparison

| Criterion | FDM | ANN | PINN |
|-----------|-----|-----|------|
| Requires simulation data | ✓ | ✓ | ✗ |
| Enforces physics | ✓ | ✗ | ✓ |
| Generalises across materials | Re-run required | Partially | ✓ |
| Inference cost | High | Low | Low |
| Interpretability | High | Low | Medium |

---
*Department of Mechanical & Aeronautical Engineering | University of Pretoria*
